# Review and combine DEC visitation files

Each source file has its own code block. Run a block, inspect its standardized preview, then set that block's `ACCEPT_...` switch to `True` to include it in the master CSV. Elm Ridge is preserved as **hourly** data; it is not silently aggregated to monthly totals.

All outputs use the same 12 columns as the existing DEC cleaned visitation records.

In [1]:
from pathlib import Path
import calendar, re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'final code' else NOTEBOOK_DIR
SOURCE_DIR = PROJECT_ROOT / 'DEC Data Records'
OUTPUT_CSV = NOTEBOOK_DIR / 'cleaned_all_visitation_records.csv'
QC_CSV = NOTEBOOK_DIR / 'cleaned_all_visitation_records_qc.csv'
COLUMNS = ['location','start_date','end_date','year','month','visitation_count','frequency','source_file','source_sheet','source_row','source_col','target_processing']
MONTHS = {n.lower(): i for i,n in enumerate(calendar.month_name) if n} | {n.lower(): i for i,n in enumerate(calendar.month_abbr) if n}
MONTHS['novemeber'] = 11
EXCLUDED_SHEETS = {'yearly totals','wilderness area','wild forest','intensive use area','sheet1','sheet2'}
accepted_sources = []
review_log = []

def clean_text(value): return re.sub(r'\s+', ' ', str(value)).strip()
def canonical_location(value):
    text = re.sub(r'\s*\(removed\)\s*', '', clean_text(value), flags=re.I)
    aliases = {'overlook mountain':'Overlook','elm ridge (peck rd)':'Elm Ridge','elm ridge hike+bike':'Elm Ridge','mt. tremper':'Mt Tremper','mount tremper':'Mt Tremper','khp':'Kaaterskill High Peak','slide mt':'Slide Mountain','slide mt.':'Slide Mountain'}
    return aliases.get(text.casefold(), text)
def numeric(value):
    if pd.isna(value) or isinstance(value, bool): return np.nan
    if isinstance(value, (int,float,np.number)): return float(value) if np.isfinite(value) else np.nan
    text = clean_text(value).replace(',','')
    return float(text) if re.fullmatch(r'[-+]?\d+(?:\.\d+)?', text) else np.nan
def year_value(value):
    m = re.fullmatch(r'(19\d{2}|20\d{2})(?:\.0)?\*?', clean_text(value))
    return int(m.group(1)) if m else None
def month_value(value): return MONTHS.get(clean_text(value).casefold())

def make_record(location, start, end, count, frequency, source_file, source_sheet, source_row, source_col, note):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    return {'location':canonical_location(location),'start_date':start,'end_date':end,'year':start.year,'month':start.month,
            'visitation_count':count,'frequency':frequency,'source_file':source_file,'source_sheet':source_sheet,
            'source_row':int(source_row),'source_col':int(source_col),'target_processing':note}

def monthly_record(location, year, month, count, *source):
    start = pd.Timestamp(year=year, month=month, day=1)
    return make_record(location, start, start + pd.offsets.MonthEnd(0), count, 'monthly', *source)
def annual_record(location, year, count, *source):
    record = make_record(location, f'{year}-01-01', f'{year}-12-31', count, 'annual', *source)
    record['month'] = pd.NA
    return record

def standardize(records):
    df = pd.DataFrame(records, columns=COLUMNS)
    if df.empty: return df
    df['visitation_count'] = pd.to_numeric(df.visitation_count, errors='coerce').round(6)
    df = df.dropna(subset=['location','start_date','end_date','visitation_count']).query('visitation_count >= 0').copy()
    return df[COLUMNS].sort_values(['location','start_date','frequency']).reset_index(drop=True)

def review_and_accept(df, label, accept):
    print(f'{label}: {len(df):,} standardized records; accepted={accept}')
    if not df.empty:
        summary = (df.groupby('location',as_index=False)
                   .agg(initial_start_date=('start_date','min'),final_end_date=('end_date','max'),
                        believed_frequency=('frequency',lambda s: ', '.join(sorted(s.unique()))),
                        average_visitors=('visitation_count','mean'),records=('visitation_count','size'))
                   .sort_values('location').reset_index(drop=True))
        summary['average_visitors'] = summary.average_visitors.round(2)
        print(f'Total locations: {len(summary):,}')
        display(summary)
        print('Standardized record preview:')
        display(df.head(10))
    review_log.append({'source_file':label,'records_standardized':len(df),'accepted':bool(accept)})
    if accept: accepted_sources.append(df.copy())

def preview_location(df, selected_location):
    locations = sorted(df.location.dropna().unique().tolist())
    print('Available locations:')
    print(locations)
    if selected_location not in locations:
        raise ValueError(f'{selected_location!r} is not available. Copy one of the location names printed above.')
    location_df = df[df.location.eq(selected_location)].sort_values(['start_date','frequency']).reset_index(drop=True)
    print(f'Previewing {selected_location}: {len(location_df):,} records')
    display(location_df)
    return location_df


In [2]:
def parse_register_matrix(path):
    records = []
    for sheet in pd.ExcelFile(path).sheet_names:
        if clean_text(sheet).casefold() in EXCLUDED_SHEETS: continue
        raw = pd.read_excel(path, sheet_name=sheet, header=None)
        if len(raw) < 16 or raw.shape[1] < 2: continue
        years = [year_value(v) for v in raw.iloc[2]]
        if sum(y is not None for y in years) < 2: continue
        for r in range(len(raw)):
            month = month_value(raw.iat[r,0])
            if not month: continue
            for c,year in enumerate(years):
                if year is None or c >= raw.shape[1]: continue
                count = numeric(raw.iat[r,c])
                if pd.notna(count) and count >= 0:
                    records.append(monthly_record(sheet,year,month,count,path.name,sheet,r+1,c+1,'observed_monthly_from_register_matrix'))
    return standardize(records)

def parse_trafx(path):
    records, raw = [], pd.read_excel(path, sheet_name='Count data')
    # The workbook uses merged/blank year cells: one printed year applies to all following site rows until the next year.
    raw['_effective_year'] = raw['Year'].ffill()
    month_cols = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
    for i,row in raw.iterrows():
        year, location = year_value(row.get('_effective_year')), row.get('Site')
        if not year or pd.isna(location): continue
        for col,month in month_cols.items():
            count = numeric(row.get(col))
            if pd.notna(count) and count >= 0:
                records.append(monthly_record(location,year,month,count,path.name,'Count data',i+2,raw.columns.get_loc(col)+1,'observed_monthly_from_trafx_summary'))
    return standardize(records)

def parse_elm_ridge_hourly(path):
    raw = pd.read_csv(path, header=None, names=['timestamp','count'])
    raw['timestamp'] = pd.to_datetime(raw.timestamp, errors='coerce')
    raw['count'] = pd.to_numeric(raw['count'], errors='coerce')
    raw['source_row'] = np.arange(1,len(raw)+1)
    raw = raw.dropna(subset=['timestamp','count']).query('count >= 0').copy()
    # If a timestamp appears more than once, sum it and retain all contributing row numbers in the note.
    records = []
    for timestamp,group in raw.groupby('timestamp', sort=True):
        rows = ','.join(group.source_row.astype(str))
        records.append(make_record('Elm Ridge',timestamp,timestamp+pd.Timedelta(hours=1)-pd.Timedelta(seconds=1),group['count'].sum(),'hourly',path.name,'CSV',group.source_row.min(),2,f'observed_hourly; contributing_source_rows={rows}'))
    return standardize(records)

def parse_canister_workbook(path):
    # Sheet1 contains repeated location/year/count blocks. Sheet3 contains location/month/count blocks.
    # Sheet2 contains an explicitly labelled all-location monthly total. Every worksheet is inspected.
    records, annual_lookup = [], {}
    excel = pd.ExcelFile(path)
    for sheet in excel.sheet_names:
        raw = pd.read_excel(path,sheet_name=sheet,header=None)
        if sheet == 'Sheet1':
            for r in range(len(raw)-1):
                candidates = [raw.iat[r,c] for c in range(min(2,raw.shape[1])) if isinstance(raw.iat[r,c],str)]
                if not candidates: continue
                label = clean_text(candidates[-1])
                label = re.sub(r'\s+2008\s*[-–]\s*2025$','',label,flags=re.I)
                label = re.sub(r'\s*\((?:not complete|not done|done)\)\s*',' ',label,flags=re.I)
                label = re.sub(r'\s+(?:not done|done)\s*$','',label,flags=re.I).strip()
                if not label or year_value(label) or 'total' in label.casefold() or month_value(label): continue
                starts = [rr for rr in range(r+1,min(r+4,len(raw))) if year_value(raw.iat[rr,0]) is not None and pd.notna(numeric(raw.iat[rr,1]))]
                if not starts: continue
                rr = starts[0]
                while rr < len(raw):
                    year,count = year_value(raw.iat[rr,0]),numeric(raw.iat[rr,1])
                    if year is None or pd.isna(count): break
                    location = canonical_location(label)
                    records.append(annual_record(location,year,count,path.name,sheet,rr+1,2,'observed_annual_from_canister_location_block'))
                    annual_lookup[(location,year)] = count
                    rr += 1

    # Parse monthly blocks after annual totals are known so unlabeled years can be inferred by matching totals.
    for sheet in excel.sheet_names:
        raw = pd.read_excel(path,sheet_name=sheet,header=None)
        if sheet == 'Sheet3':
            for r in range(len(raw)-12):
                for c in range(raw.shape[1]):
                    if not isinstance(raw.iat[r,c],str): continue
                    header = clean_text(raw.iat[r,c])
                    explicit = re.match(r'^(19\d{2}|20\d{2})\s+(.+)$',header)
                    patterns = []
                    if c+1 < raw.shape[1] and month_value(raw.iat[r+1,c]) and pd.notna(numeric(raw.iat[r+1,c+1])):
                        patterns.append((c,c+1))
                    if c>0 and month_value(raw.iat[r+1,c-1]) and pd.notna(numeric(raw.iat[r+1,c])):
                        patterns.append((c-1,c))
                    for month_col,count_col in patterns:
                        location = canonical_location(explicit.group(2) if explicit else header)
                        values = []
                        for rr in range(r+1,min(r+13,len(raw))):
                            month,count = month_value(raw.iat[rr,month_col]),numeric(raw.iat[rr,count_col])
                            if month and pd.notna(count): values.append((rr,month,count))
                        if len(values) != 12: continue
                        if explicit:
                            year = int(explicit.group(1))
                        else:
                            total = round(sum(v[2] for v in values),6)
                            matches = [y for (loc,y),count in annual_lookup.items() if loc==location and abs(count-total)<0.01]
                            if matches:
                                year = max(matches)
                            else:
                                location_years = [y for (loc,y) in annual_lookup if loc==location]
                                year = max(location_years)-1 if location_years else None
                        if year is None: continue
                        for rr,month,count in values:
                            records.append(monthly_record(location,year,month,count,path.name,sheet,rr+1,count_col+1,'observed_monthly_from_canister_location_block'))
        elif sheet == 'Sheet2':
            # Explicit sheet-wide monthly total, such as Sheet2: 2019 / Total Visitation by Month 2019.
            explicit_years = [year_value(v) for v in raw.stack().tolist()]
            explicit_years = [y for y in explicit_years if y]
            if explicit_years:
                year = explicit_years[0]
                for r in range(len(raw)):
                    month = month_value(raw.iat[r,0]) if raw.shape[1]>1 else None
                    count = numeric(raw.iat[r,1]) if raw.shape[1]>1 else np.nan
                    if month and pd.notna(count):
                        records.append(monthly_record('All Canister Peaks',year,month,count,path.name,sheet,r+1,2,'observed_monthly_all_canister_peaks'))
    return standardize(records).drop_duplicates(['location','start_date','end_date','frequency','visitation_count']).reset_index(drop=True)


## 1. 2008–2025 Total Canister Sign-In
Inspect the preview, then change the acceptance switch if needed.

In [3]:
ACCEPT_OLD_CANISTER = True  # Change to False to exclude this file from the master.
old_canister_path = SOURCE_DIR / '2008-2025 Total Canister Sign-In.xlsx'
old_canister = parse_canister_workbook(old_canister_path)
review_and_accept(old_canister, old_canister_path.name, ACCEPT_OLD_CANISTER)
SELECTED_LOCATION_OLD_CANISTER = sorted(old_canister.location.unique())[1]  # Replace with any printed location.
old_canister_location_df = preview_location(old_canister, SELECTED_LOCATION_OLD_CANISTER)

2008-2025 Total Canister Sign-In.xlsx: 407 standardized records; accepted=True
Total locations: 15


,location,initial_start_date,final_end_date,believed_frequency,average_visitors,records
0,All Canister Peaks,2019-01-01,2019-12-31,monthly,776.17,12
1,Balsam Cap,2008-01-01,2025-12-31,"annual, monthly",436.37,30
2,Big Indian,2008-01-01,2025-12-31,"annual, monthly",443.87,30
3,Eagle,2020-01-01,2025-12-31,"annual, monthly",418.11,18
4,Fir,2008-01-01,2025-12-31,"annual, monthly",388.47,30
5,Friday,2008-01-01,2025-12-31,"annual, monthly",401.17,29
6,Halcott,2008-01-01,2025-12-31,"annual, monthly",403.50,30
7,Kaaterskill High Peak,2020-01-01,2025-12-31,"annual, monthly",500.00,18
8,Lone,2008-01-01,2025-12-31,"annual, monthly",402.23,30
9,North Dome,2008-01-01,2025-12-31,"annual, monthly",379.70,30


Standardized record preview:


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,All Canister Peaks,2019-01-01,2019-01-31,2019,1,834.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,4,2,observed_monthly_all_canister_peaks
1,All Canister Peaks,2019-02-01,2019-02-28,2019,2,867.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,5,2,observed_monthly_all_canister_peaks
2,All Canister Peaks,2019-03-01,2019-03-31,2019,3,800.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,6,2,observed_monthly_all_canister_peaks
3,All Canister Peaks,2019-04-01,2019-04-30,2019,4,616.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,7,2,observed_monthly_all_canister_peaks
4,All Canister Peaks,2019-05-01,2019-05-31,2019,5,877.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,8,2,observed_monthly_all_canister_peaks
5,All Canister Peaks,2019-06-01,2019-06-30,2019,6,633.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,9,2,observed_monthly_all_canister_peaks
6,All Canister Peaks,2019-07-01,2019-07-31,2019,7,696.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,10,2,observed_monthly_all_canister_peaks
7,All Canister Peaks,2019-08-01,2019-08-31,2019,8,663.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,11,2,observed_monthly_all_canister_peaks
8,All Canister Peaks,2019-09-01,2019-09-30,2019,9,857.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,12,2,observed_monthly_all_canister_peaks
9,All Canister Peaks,2019-10-01,2019-10-31,2019,10,767.0,monthly,2008-2025 Total Canister Sign-In.xlsx,Sheet2,13,2,observed_monthly_all_canister_peaks


Available locations:
['All Canister Peaks', 'Balsam Cap', 'Big Indian', 'Eagle', 'Fir', 'Friday', 'Halcott', 'Kaaterskill High Peak', 'Lone', 'North Dome', 'Rocky', 'Rusk', 'SW Hunter', 'Sherrill', 'Vly']
Previewing Balsam Cap: 30 records


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Balsam Cap,2008-01-01,2008-12-31,2008,<NA>,179.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,10,2,observed_annual_from_canister_location_block
1,Balsam Cap,2009-01-01,2009-12-31,2009,<NA>,221.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,11,2,observed_annual_from_canister_location_block
2,Balsam Cap,2010-01-01,2010-12-31,2010,<NA>,216.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,12,2,observed_annual_from_canister_location_block
3,Balsam Cap,2011-01-01,2011-12-31,2011,<NA>,296.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,13,2,observed_annual_from_canister_location_block
4,Balsam Cap,2012-01-01,2012-12-31,2012,<NA>,426.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,14,2,observed_annual_from_canister_location_block
5,Balsam Cap,2013-01-01,2013-12-31,2013,<NA>,366.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,15,2,observed_annual_from_canister_location_block
6,Balsam Cap,2014-01-01,2014-12-31,2014,<NA>,512.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,16,2,observed_annual_from_canister_location_block
7,Balsam Cap,2015-01-01,2015-12-31,2015,<NA>,576.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,17,2,observed_annual_from_canister_location_block
8,Balsam Cap,2016-01-01,2016-12-31,2016,<NA>,681.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,18,2,observed_annual_from_canister_location_block
9,Balsam Cap,2017-01-01,2017-12-31,2017,<NA>,657.0,annual,2008-2025 Total Canister Sign-In.xlsx,Sheet1,19,2,observed_annual_from_canister_location_block


## 2. Canister sign-in charts R3 and R4 2025

In [4]:
ACCEPT_2025_CANISTER = True
canister_2025_path = SOURCE_DIR / 'Canister sign in charts R3 and R4 2025.xlsx'
canister_2025 = parse_canister_workbook(canister_2025_path)
review_and_accept(canister_2025, canister_2025_path.name, ACCEPT_2025_CANISTER)
SELECTED_LOCATION_2025_CANISTER = sorted(canister_2025.location.unique())[1]  # Replace with any printed location.
canister_2025_location_df = preview_location(canister_2025, SELECTED_LOCATION_2025_CANISTER)

Canister sign in charts R3 and R4 2025.xlsx: 431 standardized records; accepted=True
Total locations: 15


,location,initial_start_date,final_end_date,believed_frequency,average_visitors,records
0,All Canister Peaks,2019-01-01,2019-12-31,monthly,776.17,12
1,Balsam Cap,2008-01-01,2025-12-31,"annual, monthly",333.71,42
2,Big Indian,2008-01-01,2025-12-31,"annual, monthly",442.47,30
3,Eagle,2020-01-01,2025-12-31,"annual, monthly",416.72,18
4,Fir,2008-01-01,2025-12-31,"annual, monthly",386.77,30
5,Friday,2008-01-01,2025-12-31,"annual, monthly",400.86,29
6,Halcott,2008-01-01,2025-12-31,annual,621.61,18
7,Kaaterskill High Peak,2020-01-01,2025-12-31,"annual, monthly",498.67,18
8,Lone,2008-01-01,2025-12-31,"annual, monthly",308.29,42
9,North Dome,2008-01-01,2025-12-31,"annual, monthly",379.70,30


Standardized record preview:


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,All Canister Peaks,2019-01-01,2019-01-31,2019,1,834.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,4,2,observed_monthly_all_canister_peaks
1,All Canister Peaks,2019-02-01,2019-02-28,2019,2,867.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,5,2,observed_monthly_all_canister_peaks
2,All Canister Peaks,2019-03-01,2019-03-31,2019,3,800.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,6,2,observed_monthly_all_canister_peaks
3,All Canister Peaks,2019-04-01,2019-04-30,2019,4,616.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,7,2,observed_monthly_all_canister_peaks
4,All Canister Peaks,2019-05-01,2019-05-31,2019,5,877.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,8,2,observed_monthly_all_canister_peaks
5,All Canister Peaks,2019-06-01,2019-06-30,2019,6,633.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,9,2,observed_monthly_all_canister_peaks
6,All Canister Peaks,2019-07-01,2019-07-31,2019,7,696.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,10,2,observed_monthly_all_canister_peaks
7,All Canister Peaks,2019-08-01,2019-08-31,2019,8,663.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,11,2,observed_monthly_all_canister_peaks
8,All Canister Peaks,2019-09-01,2019-09-30,2019,9,857.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,12,2,observed_monthly_all_canister_peaks
9,All Canister Peaks,2019-10-01,2019-10-31,2019,10,767.0,monthly,Canister sign in charts R3 and R4 2025.xlsx,Sheet2,13,2,observed_monthly_all_canister_peaks


Available locations:
['All Canister Peaks', 'Balsam Cap', 'Big Indian', 'Eagle', 'Fir', 'Friday', 'Halcott', 'Kaaterskill High Peak', 'Lone', 'North Dome', 'Rocky', 'Rusk', 'SW Hunter', 'Sherrill', 'Vly']
Previewing Balsam Cap: 42 records


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Balsam Cap,2008-01-01,2008-12-31,2008,<NA>,179.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,10,2,observed_annual_from_canister_location_block
1,Balsam Cap,2009-01-01,2009-12-31,2009,<NA>,221.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,11,2,observed_annual_from_canister_location_block
2,Balsam Cap,2010-01-01,2010-12-31,2010,<NA>,216.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,12,2,observed_annual_from_canister_location_block
3,Balsam Cap,2011-01-01,2011-12-31,2011,<NA>,296.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,13,2,observed_annual_from_canister_location_block
4,Balsam Cap,2012-01-01,2012-12-31,2012,<NA>,426.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,14,2,observed_annual_from_canister_location_block
5,Balsam Cap,2013-01-01,2013-12-31,2013,<NA>,366.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,15,2,observed_annual_from_canister_location_block
6,Balsam Cap,2014-01-01,2014-12-31,2014,<NA>,512.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,16,2,observed_annual_from_canister_location_block
7,Balsam Cap,2015-01-01,2015-12-31,2015,<NA>,576.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,17,2,observed_annual_from_canister_location_block
8,Balsam Cap,2016-01-01,2016-12-31,2016,<NA>,681.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,18,2,observed_annual_from_canister_location_block
9,Balsam Cap,2017-01-01,2017-12-31,2017,<NA>,657.0,annual,Canister sign in charts R3 and R4 2025.xlsx,Sheet1,19,2,observed_annual_from_canister_location_block


## 3. Elm Ridge Hike + Bike RAW — hourly
Every valid source timestamp becomes an hourly record. `end_date` is 59 minutes and 59 seconds after `start_date`.

In [5]:
ACCEPT_ELM_RIDGE_HOURLY = True
elm_path = SOURCE_DIR / 'Elm Ridge Hike+Bike RAW (1).csv'
elm_hourly = parse_elm_ridge_hourly(elm_path)
review_and_accept(elm_hourly, elm_path.name, ACCEPT_ELM_RIDGE_HOURLY)
print('Hourly validation:', {'records':len(elm_hourly),'unique_start_times':elm_hourly.start_date.nunique(),'frequency_values':elm_hourly.frequency.unique().tolist()})
SELECTED_LOCATION_ELM_RIDGE = sorted(elm_hourly.location.unique())[0]  # Replace with any printed location.
elm_ridge_location_df = preview_location(elm_hourly, SELECTED_LOCATION_ELM_RIDGE)

Elm Ridge Hike+Bike RAW (1).csv: 49,033 standardized records; accepted=True
Total locations: 1


,location,initial_start_date,final_end_date,believed_frequency,average_visitors,records
0,Elm Ridge,2020-07-22 12:00:00,2026-04-01 13:59:59,hourly,2.98,49033


Standardized record preview:


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Elm Ridge,2020-07-22 12:00:00,2020-07-22 12:59:59,2020,7,16,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,1,2,observed_hourly; contributing_source_rows=1
1,Elm Ridge,2020-07-22 13:00:00,2020-07-22 13:59:59,2020,7,19,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,2,2,observed_hourly; contributing_source_rows=2
2,Elm Ridge,2020-07-22 14:00:00,2020-07-22 14:59:59,2020,7,14,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,3,2,observed_hourly; contributing_source_rows=3
3,Elm Ridge,2020-07-22 15:00:00,2020-07-22 15:59:59,2020,7,18,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,4,2,observed_hourly; contributing_source_rows=4
4,Elm Ridge,2020-07-22 16:00:00,2020-07-22 16:59:59,2020,7,4,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,5,2,observed_hourly; contributing_source_rows=5
5,Elm Ridge,2020-07-22 17:00:00,2020-07-22 17:59:59,2020,7,4,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,6,2,observed_hourly; contributing_source_rows=6
6,Elm Ridge,2020-07-22 18:00:00,2020-07-22 18:59:59,2020,7,1,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,7,2,observed_hourly; contributing_source_rows=7
7,Elm Ridge,2020-07-22 19:00:00,2020-07-22 19:59:59,2020,7,1,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,8,2,observed_hourly; contributing_source_rows=8
8,Elm Ridge,2020-07-22 20:00:00,2020-07-22 20:59:59,2020,7,0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,9,2,observed_hourly; contributing_source_rows=9
9,Elm Ridge,2020-07-22 21:00:00,2020-07-22 21:59:59,2020,7,0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,10,2,observed_hourly; contributing_source_rows=10


Hourly validation: {'records': 49033, 'unique_start_times': 49033, 'frequency_values': ['hourly']}
Available locations:
['Elm Ridge']
Previewing Elm Ridge: 49,033 records


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Elm Ridge,2020-07-22 12:00:00,2020-07-22 12:59:59,2020,7,16,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,1,2,observed_hourly; contributing_source_rows=1
1,Elm Ridge,2020-07-22 13:00:00,2020-07-22 13:59:59,2020,7,19,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,2,2,observed_hourly; contributing_source_rows=2
2,Elm Ridge,2020-07-22 14:00:00,2020-07-22 14:59:59,2020,7,14,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,3,2,observed_hourly; contributing_source_rows=3
3,Elm Ridge,2020-07-22 15:00:00,2020-07-22 15:59:59,2020,7,18,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,4,2,observed_hourly; contributing_source_rows=4
4,Elm Ridge,2020-07-22 16:00:00,2020-07-22 16:59:59,2020,7,4,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,5,2,observed_hourly; contributing_source_rows=5
...,...,...,...,...,...,...,...,...,...,...,...,...
49028,Elm Ridge,2026-04-01 09:00:00,2026-04-01 09:59:59,2026,4,2,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,49029,2,observed_hourly; contributing_source_rows=49029
49029,Elm Ridge,2026-04-01 10:00:00,2026-04-01 10:59:59,2026,4,2,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,49030,2,observed_hourly; contributing_source_rows=49030
49030,Elm Ridge,2026-04-01 11:00:00,2026-04-01 11:59:59,2026,4,0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,49031,2,observed_hourly; contributing_source_rows=49031
49031,Elm Ridge,2026-04-01 12:00:00,2026-04-01 12:59:59,2026,4,2,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,49032,2,observed_hourly; contributing_source_rows=49032


## 4. TRAFx Master Summary

In [6]:
ACCEPT_TRAFX = True
trafx_path = SOURCE_DIR / 'TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx'
trafx = parse_trafx(trafx_path)
review_and_accept(trafx, trafx_path.name, ACCEPT_TRAFX)
SELECTED_LOCATION_TRAFX = sorted(trafx.location.unique())[6]  # Replace with any printed location.
trafx_location_df = preview_location(trafx, SELECTED_LOCATION_TRAFX)

TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx: 100 standardized records; accepted=True
Total locations: 8


,location,initial_start_date,final_end_date,believed_frequency,average_visitors,records
0,Balsam Lake Mountain,2026-01-01,2026-03-31,monthly,161.06,3
1,Giant Ledge,2025-02-01,2026-03-31,monthly,1135.57,14
2,Kanape Brook,2025-09-01,2026-03-31,monthly,183.82,7
3,Lost Clove,2025-04-01,2026-03-31,monthly,21.75,12
4,Moon Haw Road,2025-09-01,2026-03-31,monthly,86.55,7
5,Mt Tremper,2025-09-01,2026-03-31,monthly,331.18,7
6,Overlook,2021-07-01,2026-03-31,monthly,2646.41,36
7,Slide Mountain,2025-02-01,2026-03-31,monthly,324.29,14


Standardized record preview:


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Balsam Lake Mountain,2026-01-01,2026-01-31,2026,1,149.1875,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,12,3,observed_monthly_from_trafx_summary
1,Balsam Lake Mountain,2026-02-01,2026-02-28,2026,2,210.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,12,4,observed_monthly_from_trafx_summary
2,Balsam Lake Mountain,2026-03-01,2026-03-31,2026,3,124.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,12,5,observed_monthly_from_trafx_summary
3,Giant Ledge,2025-02-01,2025-02-28,2025,2,0.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,5,4,observed_monthly_from_trafx_summary
4,Giant Ledge,2025-03-01,2025-03-31,2025,3,0.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,5,5,observed_monthly_from_trafx_summary
5,Giant Ledge,2025-04-01,2025-04-30,2025,4,753.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,5,6,observed_monthly_from_trafx_summary
6,Giant Ledge,2025-05-01,2025-05-31,2025,5,1379.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,5,7,observed_monthly_from_trafx_summary
7,Giant Ledge,2025-06-01,2025-06-30,2025,6,1384.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,5,8,observed_monthly_from_trafx_summary
8,Giant Ledge,2025-07-01,2025-07-31,2025,7,1809.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,5,9,observed_monthly_from_trafx_summary
9,Giant Ledge,2025-08-01,2025-08-31,2025,8,2632.0000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,5,10,observed_monthly_from_trafx_summary


Available locations:
['Balsam Lake Mountain', 'Giant Ledge', 'Kanape Brook', 'Lost Clove', 'Moon Haw Road', 'Mt Tremper', 'Overlook', 'Slide Mountain']
Previewing Overlook: 36 records


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Overlook,2021-07-01,2021-07-31,2021,7,8060.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,2,9,observed_monthly_from_trafx_summary
1,Overlook,2021-08-01,2021-08-31,2021,8,3146.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,2,10,observed_monthly_from_trafx_summary
2,Overlook,2021-09-01,2021-09-30,2021,9,3430.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,2,11,observed_monthly_from_trafx_summary
3,Overlook,2021-10-01,2021-10-31,2021,10,4493.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,2,12,observed_monthly_from_trafx_summary
4,Overlook,2021-11-01,2021-11-30,2021,11,2785.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,2,13,observed_monthly_from_trafx_summary
5,Overlook,2021-12-01,2021-12-31,2021,12,1679.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,2,14,observed_monthly_from_trafx_summary
6,Overlook,2022-01-01,2022-01-31,2022,1,1175.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,3,3,observed_monthly_from_trafx_summary
7,Overlook,2022-02-01,2022-02-28,2022,2,1257.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,3,4,observed_monthly_from_trafx_summary
8,Overlook,2022-03-01,2022-03-31,2022,3,1553.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,3,5,observed_monthly_from_trafx_summary
9,Overlook,2022-04-01,2022-04-30,2022,4,2757.000000,monthly,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,Count data,3,6,observed_monthly_from_trafx_summary


## 5. Trail Register Database R4

In [7]:
ACCEPT_REGISTER_DATABASE_R4 = True
register_r4_path = SOURCE_DIR / 'Trail Register Database R4(1).xlsx'
register_r4 = parse_register_matrix(register_r4_path)
review_and_accept(register_r4, register_r4_path.name, ACCEPT_REGISTER_DATABASE_R4)
SELECTED_LOCATION_REGISTER_R4 = sorted(register_r4.location.unique())[0]  # Replace with any printed location.
register_r4_location_df = preview_location(register_r4, SELECTED_LOCATION_REGISTER_R4)

Trail Register Database R4(1).xlsx: 13,183 standardized records; accepted=True
Total locations: 48


,location,initial_start_date,final_end_date,believed_frequency,average_visitors,records
0,Acra Point,1996-01-01,2025-11-30,monthly,96.26,298
1,Barnum Road,1996-01-01,2025-07-31,monthly,97.18,292
2,Batavia Kill,1996-01-01,2025-11-30,monthly,223.33,307
3,Becker Hollow,1996-01-01,2025-08-31,monthly,149.83,298
4,Beech Hill Rd,1997-01-01,2026-01-31,monthly,15.53,297
5,Big Pond East,1997-04-01,2026-01-31,monthly,33.92,296
6,Big Pond West,1997-01-01,2026-01-31,monthly,20.83,299
7,Bouchoux Trail,2000-01-01,2025-12-31,monthly,122.86,266
8,Boulder Rock,1996-03-01,2019-10-31,monthly,219.21,222
9,Campbell Brook,2004-01-01,2025-12-31,monthly,14.82,233


Standardized record preview:


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Acra Point,1996-01-01,1996-01-31,1996,1,5.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,5,8,observed_monthly_from_register_matrix
1,Acra Point,1996-02-01,1996-02-29,1996,2,18.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,6,8,observed_monthly_from_register_matrix
2,Acra Point,1996-03-01,1996-03-31,1996,3,44.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,7,8,observed_monthly_from_register_matrix
3,Acra Point,1996-04-01,1996-04-30,1996,4,85.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,8,8,observed_monthly_from_register_matrix
4,Acra Point,1996-05-01,1996-05-31,1996,5,100.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,9,8,observed_monthly_from_register_matrix
5,Acra Point,1996-06-01,1996-06-30,1996,6,172.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,10,8,observed_monthly_from_register_matrix
6,Acra Point,1996-07-01,1996-07-31,1996,7,130.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,11,8,observed_monthly_from_register_matrix
7,Acra Point,1996-08-01,1996-08-31,1996,8,128.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,12,8,observed_monthly_from_register_matrix
8,Acra Point,1997-01-01,1997-01-31,1997,1,18.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,5,9,observed_monthly_from_register_matrix
9,Acra Point,1997-02-01,1997-02-28,1997,2,36.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,6,9,observed_monthly_from_register_matrix


Available locations:
['Acra Point', 'Barnum Road', 'Batavia Kill', 'Becker Hollow', 'Beech Hill Rd', 'Big Pond East', 'Big Pond West', 'Bouchoux Trail', 'Boulder Rock', 'Campbell Brook', 'Campbell Mt', 'Catskill Mt. House', 'Colgate Lake (Dutcher Notch)', "Colonel's Chair", 'DRB Margaretville Pakatakan', 'Devils Tombstone (Hunter Mtn)', 'Devils Tombstone (Plateau Mtn)', 'Diamond Notch (Lanesville)', 'Diamond Notch (Spruceton)', 'East Windham', 'Elm Ridge', 'German Hollow', 'Harding Road Trail', 'Hill Road (Huckleberry Loop N)', 'Holiday & Berry Brook Road', 'Huckleberry Loop Trail', 'Huggins Lake Trail', 'Kaaterskill Falls', "Layman's Monument", 'Little Pond Campground', 'Little Pond Loop', 'Long Path North', 'Mary Smith Hill Road', "Mary's Glen", 'Mink Hollow (Lake Hill)', 'Mountain TPKE Horse Trail', 'Mud Pond Trail', 'Ploutz Rd (Huckleberry Loop S)', 'Prediger Road', 'Roaring Kill', 'Rock Shelter (N Lake Bypass)', 'Rt 206 Cat Hollow', 'Russell Brook', 'S.Mtn Esc Trl (Schutt Rd Trail

,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Acra Point,1996-01-01,1996-01-31,1996,1,5.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,5,8,observed_monthly_from_register_matrix
1,Acra Point,1996-02-01,1996-02-29,1996,2,18.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,6,8,observed_monthly_from_register_matrix
2,Acra Point,1996-03-01,1996-03-31,1996,3,44.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,7,8,observed_monthly_from_register_matrix
3,Acra Point,1996-04-01,1996-04-30,1996,4,85.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,8,8,observed_monthly_from_register_matrix
4,Acra Point,1996-05-01,1996-05-31,1996,5,100.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,9,8,observed_monthly_from_register_matrix
...,...,...,...,...,...,...,...,...,...,...,...,...
293,Acra Point,2025-07-01,2025-07-31,2025,7,157.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,11,37,observed_monthly_from_register_matrix
294,Acra Point,2025-08-01,2025-08-31,2025,8,180.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,12,37,observed_monthly_from_register_matrix
295,Acra Point,2025-09-01,2025-09-30,2025,9,173.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,13,37,observed_monthly_from_register_matrix
296,Acra Point,2025-10-01,2025-10-31,2025,10,174.0,monthly,Trail Register Database R4(1).xlsx,Acra Point,14,37,observed_monthly_from_register_matrix


## 6. Trail Register Tally 8/9/24

In [8]:
ACCEPT_REGISTER_TALLY = True
register_tally_path = SOURCE_DIR / 'Trail Register Tally_8_9_24_(1).xlsx'
register_tally = parse_register_matrix(register_tally_path)
review_and_accept(register_tally, register_tally_path.name, ACCEPT_REGISTER_TALLY)
SELECTED_LOCATION_REGISTER_TALLY = sorted(register_tally.location.unique())[0]  # Replace with any printed location.
register_tally_location_df = preview_location(register_tally, SELECTED_LOCATION_REGISTER_TALLY)

Trail Register Tally_8_9_24_(1).xlsx: 21,015 standardized records; accepted=True
Total locations: 35


,location,initial_start_date,final_end_date,believed_frequency,average_visitors,records
0,Alder Lake,1991-04-01,2033-10-31,monthly,144.09,689
1,Ashokan HP,1996-01-01,2034-10-31,monthly,141.52,656
2,Bangle Hill,1996-01-01,2032-05-31,monthly,99.77,466
3,Beaverkill Road to BLM FT,1994-07-01,2033-10-31,monthly,110.38,650
4,Biscuit Brook,1991-01-01,2032-08-31,monthly,81.44,652
5,Cathedral Glen,2021-09-01,2033-09-30,monthly,10.94,182
6,Denning,1991-01-01,2033-11-30,monthly,168.31,694
7,Dry Brook Ridge to Lean-to,1994-05-01,2033-09-30,monthly,21.46,660
8,Flynn,1994-08-01,2033-10-31,monthly,44.06,637
9,Fox Hollow,1991-01-01,2033-09-30,monthly,73.04,690


Standardized record preview:


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Alder Lake,1991-04-01,1991-04-30,1991,4,144.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,8,2,observed_monthly_from_register_matrix
1,Alder Lake,1991-05-01,1991-05-31,1991,5,218.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,9,2,observed_monthly_from_register_matrix
2,Alder Lake,1991-06-01,1991-06-30,1991,6,165.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,10,2,observed_monthly_from_register_matrix
3,Alder Lake,1991-07-01,1991-07-31,1991,7,320.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,11,2,observed_monthly_from_register_matrix
4,Alder Lake,1991-08-01,1991-08-31,1991,8,216.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,12,2,observed_monthly_from_register_matrix
5,Alder Lake,1991-09-01,1991-09-30,1991,9,118.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,13,2,observed_monthly_from_register_matrix
6,Alder Lake,1991-10-01,1991-10-31,1991,10,83.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,14,2,observed_monthly_from_register_matrix
7,Alder Lake,1991-11-01,1991-11-30,1991,11,77.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,15,2,observed_monthly_from_register_matrix
8,Alder Lake,1991-12-01,1991-12-31,1991,12,7.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,16,2,observed_monthly_from_register_matrix
9,Alder Lake,1992-01-01,1992-01-31,1992,1,34.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,5,3,observed_monthly_from_register_matrix


Available locations:
['Alder Lake', 'Ashokan HP', 'Bangle Hill', 'Beaverkill Road to BLM FT', 'Biscuit Brook', 'Cathedral Glen', 'Denning', 'Dry Brook Ridge to Lean-to', 'Flynn', 'Fox Hollow', 'Frick Pond', 'Giant Ledge', 'Hardenburgh- MH Trail', 'Kelly Hollow', 'Lane Street', 'Long Pond', 'McKenley Hollow', 'Millbrook to BLM FT', 'Mongaup Pond', 'Mt Tremper', 'Neversink Hardenburgh Trail', 'Onteora Lake', 'Overlook', 'Peekamoose', 'Red Hill Dinch Road', 'Rider Hollow', 'Rochester Hollow', 'Seager', 'Slide Mountain', 'Trails End Road', 'Upper Cherrytown Road', 'Vernooy', 'Wittenberg', 'Woodchuck Hollow', 'Woodland Valley']
Previewing Alder Lake: 689 records


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
0,Alder Lake,1991-04-01,1991-04-30,1991,4,144.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,8,2,observed_monthly_from_register_matrix
1,Alder Lake,1991-05-01,1991-05-31,1991,5,218.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,9,2,observed_monthly_from_register_matrix
2,Alder Lake,1991-06-01,1991-06-30,1991,6,165.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,10,2,observed_monthly_from_register_matrix
3,Alder Lake,1991-07-01,1991-07-31,1991,7,320.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,11,2,observed_monthly_from_register_matrix
4,Alder Lake,1991-08-01,1991-08-31,1991,8,216.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,12,2,observed_monthly_from_register_matrix
...,...,...,...,...,...,...,...,...,...,...,...,...
684,Alder Lake,2033-06-01,2033-06-30,2033,6,168.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,10,95,observed_monthly_from_register_matrix
685,Alder Lake,2033-07-01,2033-07-31,2033,7,299.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,11,95,observed_monthly_from_register_matrix
686,Alder Lake,2033-08-01,2033-08-31,2033,8,303.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,12,95,observed_monthly_from_register_matrix
687,Alder Lake,2033-09-01,2033-09-30,2033,9,229.0,monthly,Trail Register Tally_8_9_24_(1).xlsx,Alder Lake,13,95,observed_monthly_from_register_matrix


## Build the master from accepted files
Rows with the same location, time range, and frequency are reconciled by source priority. Hourly and monthly observations are deliberately kept as different frequencies.

In [9]:
if not accepted_sources:
    raise ValueError('No files accepted. Set at least one ACCEPT_... switch to True and rerun its review block.')
all_rows = pd.concat(accepted_sources, ignore_index=True)
current_month = pd.Timestamp.today().to_period('M').start_time
all_rows = all_rows[(all_rows.frequency != 'monthly') | (all_rows.start_date <= current_month)].copy()
priority_patterns = [('Trail Register Database R4',50),('TRAFx',45),('Canister sign in charts',40),('Elm Ridge',35),('Trail Register Tally',30),('2008-2025 Total Canister',20)]
all_rows['_priority'] = all_rows.source_file.map(lambda x: next((p for text,p in priority_patterns if text in x),0))
key = ['location','start_date','end_date','frequency']
all_rows = all_rows.sort_values(key+['_priority','source_file'],ascending=[True,True,True,True,False,True])
def reconcile(group):
    winner = group.iloc[0].copy()
    if len(group)>1:
        status = 'duplicate_agreement' if group.visitation_count.nunique()==1 else 'overlap_conflict_preferred_highest_priority'
        winner['target_processing'] += f" | {status}; considered: {'; '.join(group.source_file.drop_duplicates())}"
    return winner
cleaned = all_rows.groupby(key,dropna=False,group_keys=False).apply(reconcile).reset_index(drop=True)
cleaned = cleaned.drop(columns=['_priority']).sort_values(['location','start_date','frequency']).reset_index(drop=True)
cleaned['year'] = cleaned.year.astype('Int64'); cleaned['month'] = cleaned.month.astype('Int64')
# Preserve timestamps for hourly rows; use date-only ISO text for monthly/annual rows.
cleaned['start_date'] = cleaned.apply(lambda r: pd.Timestamp(r.start_date).strftime('%Y-%m-%d %H:%M:%S') if r.frequency=='hourly' else pd.Timestamp(r.start_date).strftime('%Y-%m-%d'),axis=1)
cleaned['end_date'] = cleaned.apply(lambda r: pd.Timestamp(r.end_date).strftime('%Y-%m-%d %H:%M:%S') if r.frequency=='hourly' else pd.Timestamp(r.end_date).strftime('%Y-%m-%d'),axis=1)
cleaned[COLUMNS].to_csv(OUTPUT_CSV,index=False)
qc = pd.DataFrame(review_log); qc.loc[len(qc)] = {'source_file':'TOTAL CLEANED AFTER RECONCILIATION','records_standardized':len(cleaned),'accepted':True}; qc.to_csv(QC_CSV,index=False)
print(f'Wrote {len(cleaned):,} records to {OUTPUT_CSV}')
display(qc)
display(cleaned.groupby('frequency').agg(records=('location','size'),locations=('location','nunique')))
display(cleaned[cleaned.frequency=='hourly'].head(10))

/var/folders/3n/hr91kwpn3276y1btb7kd4g300000gn/T/ipykernel_64300/1917046067.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cleaned = all_rows.groupby(key,dropna=False,group_keys=False).apply(reconcile).reset_index(drop=True)


Wrote 75,626 records to /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/final code/cleaned_all_visitation_records.csv


,source_file,records_standardized,accepted
0,2008-2025 Total Canister Sign-In.xlsx,407,True
1,Canister sign in charts R3 and R4 2025.xlsx,431,True
2,Elm Ridge Hike+Bike RAW (1).csv,49033,True
3,TRAFx+Master+Summary+(2021-2026) 4_22_26.xlsx,100,True
4,Trail Register Database R4(1).xlsx,13183,True
5,Trail Register Tally_8_9_24_(1).xlsx,21015,True
6,TOTAL CLEANED AFTER RECONCILIATION,75626,True


,records,locations
frequency,,
annual,227,14
hourly,49033,1
monthly,26366,102


,location,start_date,end_date,year,month,visitation_count,frequency,source_file,source_sheet,source_row,source_col,target_processing
8458,Elm Ridge,2020-07-22 12:00:00,2020-07-22 12:59:59,2020,7,16.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,1,2,observed_hourly; contributing_source_rows=1
8459,Elm Ridge,2020-07-22 13:00:00,2020-07-22 13:59:59,2020,7,19.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,2,2,observed_hourly; contributing_source_rows=2
8460,Elm Ridge,2020-07-22 14:00:00,2020-07-22 14:59:59,2020,7,14.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,3,2,observed_hourly; contributing_source_rows=3
8461,Elm Ridge,2020-07-22 15:00:00,2020-07-22 15:59:59,2020,7,18.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,4,2,observed_hourly; contributing_source_rows=4
8462,Elm Ridge,2020-07-22 16:00:00,2020-07-22 16:59:59,2020,7,4.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,5,2,observed_hourly; contributing_source_rows=5
8463,Elm Ridge,2020-07-22 17:00:00,2020-07-22 17:59:59,2020,7,4.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,6,2,observed_hourly; contributing_source_rows=6
8464,Elm Ridge,2020-07-22 18:00:00,2020-07-22 18:59:59,2020,7,1.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,7,2,observed_hourly; contributing_source_rows=7
8465,Elm Ridge,2020-07-22 19:00:00,2020-07-22 19:59:59,2020,7,1.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,8,2,observed_hourly; contributing_source_rows=8
8466,Elm Ridge,2020-07-22 20:00:00,2020-07-22 20:59:59,2020,7,0.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,9,2,observed_hourly; contributing_source_rows=9
8467,Elm Ridge,2020-07-22 21:00:00,2020-07-22 21:59:59,2020,7,0.0,hourly,Elm Ridge Hike+Bike RAW (1).csv,CSV,10,2,observed_hourly; contributing_source_rows=10


## Conflict-free and single-frequency ML exports

A key is `location + start_date + end_date + frequency`. Agreeing duplicates collapse to one row. If repeated keys contain different visitation values, the entire conflicting key is excluded from `DEC data no_conflicts clean.csv` rather than silently choosing one value.

Aggregation follows a native-frequency-first hierarchy: native daily values replace hourly daily sums; native monthly values replace daily/hourly monthly sums; native annual values replace lower-frequency yearly sums.

In [10]:
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
NO_CONFLICTS_CSV = DATA_DIR / 'DEC data no_conflicts clean.csv'

# Remove every exact time/location/frequency key whose reported visitation values disagree.
conflict_key = ['location','start_date','end_date','frequency']
working = all_rows.copy()
working['_value_variants'] = working.groupby(conflict_key,dropna=False).visitation_count.transform('nunique')
conflicting_candidates = working[working._value_variants.gt(1)].copy()
safe_candidates = working[working._value_variants.eq(1)].copy()

def collapse_agreeing_duplicates(group):
    row = group.iloc[0].copy()
    if len(group)>1:
        row['target_processing'] += f" | duplicate_agreement_collapsed; considered: {'; '.join(group.source_file.drop_duplicates())}"
    return row

no_conflicts = (safe_candidates.groupby(conflict_key,dropna=False,group_keys=False)
                .apply(collapse_agreeing_duplicates).reset_index(drop=True))
no_conflicts = no_conflicts.drop(columns=['_priority','_value_variants'],errors='ignore').sort_values(['location','start_date','frequency']).reset_index(drop=True)
no_conflicts['year'] = no_conflicts.year.astype('Int64')
no_conflicts['month'] = no_conflicts.month.astype('Int64')

def format_dates_for_csv(df):
    result = df.copy()
    result['start_date'] = result.apply(lambda r: pd.Timestamp(r.start_date).strftime('%Y-%m-%d %H:%M:%S') if r.frequency=='hourly' else pd.Timestamp(r.start_date).strftime('%Y-%m-%d'),axis=1)
    result['end_date'] = result.apply(lambda r: pd.Timestamp(r.end_date).strftime('%Y-%m-%d %H:%M:%S') if r.frequency=='hourly' else pd.Timestamp(r.end_date).strftime('%Y-%m-%d'),axis=1)
    return result[COLUMNS]

format_dates_for_csv(no_conflicts).to_csv(NO_CONFLICTS_CSV,index=False)
print(f'Excluded {conflicting_candidates[conflict_key].drop_duplicates().shape[0]:,} conflicting keys containing {len(conflicting_candidates):,} candidate rows.')
print(f'Wrote {len(no_conflicts):,} conflict-free records to {NO_CONFLICTS_CSV}')

def aggregate_frequency(source_df, target_frequency):
    if source_df.empty: return pd.DataFrame(columns=COLUMNS)
    temp = source_df.copy()
    temp['start_date'] = pd.to_datetime(temp.start_date)
    if target_frequency=='daily':
        temp['_period'] = temp.start_date.dt.to_period('D')
    elif target_frequency=='monthly':
        temp['_period'] = temp.start_date.dt.to_period('M')
    elif target_frequency=='annual':
        temp['_period'] = temp.start_date.dt.to_period('Y')
    else: raise ValueError(target_frequency)
    rows = []
    for (location,period),group in temp.groupby(['location','_period'],sort=True):
        start = period.start_time
        end = period.end_time.floor('s')
        sources = '; '.join(sorted(group.source_file.drop_duplicates()))
        rows.append(make_record(location,start,end,group.visitation_count.sum(),target_frequency,sources,'aggregated',pd.NA if False else 0,0,f'aggregated_{target_frequency}_from_{"+".join(sorted(group.frequency.unique()))}; source_records={len(group)}'))
    result = standardize(rows)
    if target_frequency=='annual' and not result.empty: result['month'] = pd.NA
    return result

def prefer_native(native_df, aggregated_df, frequency):
    native = native_df.copy()
    if aggregated_df.empty: return native.sort_values(['location','start_date']).reset_index(drop=True)
    native_keys = set(zip(native.location,pd.to_datetime(native.start_date).dt.to_period({'daily':'D','monthly':'M','annual':'Y'}[frequency])))
    aggregate_periods = pd.to_datetime(aggregated_df.start_date).dt.to_period({'daily':'D','monthly':'M','annual':'Y'}[frequency])
    keep = [(loc,period) not in native_keys for loc,period in zip(aggregated_df.location,aggregate_periods)]
    return pd.concat([native,aggregated_df.loc[keep]],ignore_index=True).sort_values(['location','start_date']).reset_index(drop=True)

# Hourly: native hourly observations only; no lower-frequency values are disaggregated.
hourly_ml = no_conflicts[no_conflicts.frequency.eq('hourly')].copy()

# Daily: use native daily records; otherwise sum hourly records for that location/day.
native_daily = no_conflicts[no_conflicts.frequency.eq('daily')].copy()
daily_from_hourly = aggregate_frequency(hourly_ml,'daily')
daily_ml = prefer_native(native_daily,daily_from_hourly,'daily')

# Monthly: native monthly observations win. Only months without native monthly data use daily/hourly aggregation.
native_monthly = no_conflicts[no_conflicts.frequency.eq('monthly')].copy()
monthly_from_daily = aggregate_frequency(daily_ml,'monthly')
monthly_ml = prefer_native(native_monthly,monthly_from_daily,'monthly')

# Yearly: native annual observations win. Only years without native annual data use monthly aggregation.
native_annual = no_conflicts[no_conflicts.frequency.eq('annual')].copy()
yearly_from_monthly = aggregate_frequency(monthly_ml,'annual')
yearly_ml = prefer_native(native_annual,yearly_from_monthly,'annual')

frequency_outputs = {'hourly':hourly_ml,'daily':daily_ml,'monthly':monthly_ml,'yearly':yearly_ml}
for name,df in frequency_outputs.items():
    path = DATA_DIR / f'{name}.csv'
    format_dates_for_csv(df).to_csv(path,index=False)
    print(f'Wrote {len(df):,} {name} records to {path}')
display(pd.DataFrame([{'file':f'{name}.csv','records':len(df),'locations':df.location.nunique()} for name,df in frequency_outputs.items()]))

/var/folders/3n/hr91kwpn3276y1btb7kd4g300000gn/T/ipykernel_64300/2803204541.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_agreeing_duplicates).reset_index(drop=True))


Excluded 5,147 conflicting keys containing 10,331 candidate rows.
Wrote 70,479 conflict-free records to /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/data/DEC data no_conflicts clean.csv
Wrote 49,033 hourly records to /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/data/hourly.csv
Wrote 2,051 daily records to /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/data/daily.csv
Wrote 21,240 monthly records to /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/data/monthly.csv
Wrote 2,148 yearly records to /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/data/yearly.csv


,file,records,locations
0,hourly.csv,49033,1
1,daily.csv,2051,1
2,monthly.csv,21240,102
3,yearly.csv,2148,102
